In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

In [2]:
import os

In [3]:
os.chdir("C:/Users/INDUS/Desktop")

In [4]:
df = pd.read_csv("seattle-weather.csv")
df.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [5]:
X = df[["precipitation", "temp_max", "temp_min", "wind"]]
y = df["weather"]

In [6]:
df = df.drop(columns=["date"])
print(df["weather"].value_counts())
df.head()

weather
rain       641
sun        640
fog        101
drizzle     53
snow        26
Name: count, dtype: int64


,precipitation,temp_max,temp_min,wind,weather
0,0.0,12.8,5.0,4.7,drizzle
1,10.9,10.6,2.8,4.5,rain
2,0.8,11.7,7.2,2.3,rain
3,20.3,12.2,5.6,4.7,rain
4,1.3,8.9,2.8,6.1,rain


In [7]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
# KNN Model
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

knn_preds = knn.predict(X_test_scaled)

print("KNN Accuracy:", round(accuracy_score(y_test, knn_preds) * 100, 1), "%")

KNN Accuracy: 74.1 %


In [10]:
# Random Forest Model
rf = RandomForestClassifier( n_estimators=100, random_state=42)

rf.fit(X_train, y_train)

rf_preds = rf.predict(X_test)

print("Random Forest Accuracy:",round(accuracy_score(y_test, rf_preds) * 100, 1), "%")

Random Forest Accuracy: 81.6 %


In [11]:
# Predict a new day
# [precipitation, temp_max, temp_min, wind]
new_day = pd.DataFrame(
    [[5.0, 10.0, 4.0, 3.5]],
    columns=["precipitation", "temp_max", "temp_min", "wind"]
)

In [12]:
# KNN prediction
new_day_scaled = scaler.transform(new_day)
knn_result = knn.predict(new_day_scaled)[0]

In [13]:
# Random Forest prediction
rf_result = rf.predict(new_day)[0]

In [14]:
print(" WeatherCast AI ")
print("KNN Prediction          :", knn_result)
print("Random Forest Prediction:", rf_result)

 WeatherCast AI 
KNN Prediction          : rain
Random Forest Prediction: rain


In [15]:
import gradio as gr

In [16]:
# Prediction function
def predict_weather(precipitation, temp_max, temp_min, wind):

    # Create dataframe for new input
    new_day = pd.DataFrame(
        [[precipitation, temp_max, temp_min, wind]],
        columns=["precipitation", "temp_max", "temp_min", "wind"]
    )

    # KNN prediction
    new_day_scaled = scaler.transform(new_day)
    knn_prediction = knn.predict(new_day_scaled)[0]

    # Random Forest prediction
    rf_prediction = rf.predict(new_day)[0]

    # Final prediction (using Random Forest)
    final_prediction = rf_prediction

    return (
        knn_prediction,
        rf_prediction,
        final_prediction
    )


# Create Gradio app
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 🌦 WeatherCast AI
        ### Weather Prediction System using KNN and Random Forest
        Enter the weather parameters below to predict the weather condition.
        """
    )

    with gr.Row():
        precipitation = gr.Slider(
            minimum=0,
            maximum=60,
            value=5,
            step=0.1,
            label="🌧 Precipitation (mm)"
        )

        temp_max = gr.Slider(
            minimum=-5,
            maximum=40,
            value=20,
            step=0.1,
            label="🌡 Maximum Temperature (°C)"
        )

    with gr.Row():
        temp_min = gr.Slider(
            minimum=-10,
            maximum=25,
            value=10,
            step=0.1,
            label="❄ Minimum Temperature (°C)"
        )

        wind = gr.Slider(
            minimum=0,
            maximum=15,
            value=3,
            step=0.1,
            label="💨 Wind Speed (mph)"
        )

    predict_button = gr.Button("🔮 Predict Weather")

    with gr.Row():
        knn_output = gr.Textbox(label="KNN Prediction")
        rf_output = gr.Textbox(label="Random Forest Prediction")
        final_output = gr.Textbox(label="Final Prediction")

    predict_button.click(
        fn=predict_weather,
        inputs=[precipitation, temp_max, temp_min, wind],
        outputs=[knn_output, rf_output, final_output]
    )

    

demo.launch()

C:\Users\INDUS\AppData\Local\Temp\ipykernel_6956\2739938083.py:28: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
